# Tensor shape operations: `transpose`, `squeeze`, `unsqueeze`, and the `dim` / `axis` argument

A hands-on tour of the shape-manipulation tools you keep meeting in PyTorch code
(including the multi-head attention class). Run each cell top to bottom.

**Requires:** `torch`. Install with `pip install torch` if needed.

A note on naming: NumPy calls the axis argument `axis`; PyTorch calls it `dim`.
They mean the same thing — *which axis of the tensor to operate along* — and
`-1` always refers to the **last** axis.

## 0. Setup

In [2]:
import torch

def show(name, t):
    print(name)
    print(t)
    print(t.shape)
    print()

## 1. Shape and axes

Every tensor has a shape (a tuple of axis sizes) and a number of dimensions
(`ndim`). Axes are numbered `0, 1, 2, ...` from the left, or `-1, -2, ...`
from the right. The operations below all just rearrange or relabel these axes.

In [3]:
x = torch.arange(12).reshape( 3, 4)

print("Original tensor:")
print(x)


Original tensor:
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])


In [4]:
print("shape:", tuple(x.shape))   # (2, 3, 4)
print("ndim :", x.ndim)           # 3
print("axis 0 size =", x.size(0), "| axis -1 size =", x.size(-1))

shape: (3, 4)
ndim : 2
axis 0 size = 3 | axis -1 size = 4


In [5]:
x2 = torch.arange(64).reshape(2,2,2,2,2,2) 
print(x2)

tensor([[[[[[ 0,  1],
            [ 2,  3]],

           [[ 4,  5],
            [ 6,  7]]],


          [[[ 8,  9],
            [10, 11]],

           [[12, 13],
            [14, 15]]]],



         [[[[16, 17],
            [18, 19]],

           [[20, 21],
            [22, 23]]],


          [[[24, 25],
            [26, 27]],

           [[28, 29],
            [30, 31]]]]],




        [[[[[32, 33],
            [34, 35]],

           [[36, 37],
            [38, 39]]],


          [[[40, 41],
            [42, 43]],

           [[44, 45],
            [46, 47]]]],



         [[[[48, 49],
            [50, 51]],

           [[52, 53],
            [54, 55]]],


          [[[56, 57],
            [58, 59]],

           [[60, 61],
            [62, 63]]]]]])


## 2. `transpose(dim0, dim1)` — swap two axes

`transpose` swaps exactly two axes and returns a **view**: the numbers stay put
in memory, only the strides change. That is why the result is non-contiguous and
why you then need `.reshape` (not `.view`) if you want to flatten it.

For swapping *more than two* axes at once, use `permute`.

In [6]:
x = torch.arange(24).reshape(2, 3, 4)
print("x", x)
print()
xt = x.transpose(1, 2)            # swap axis 1 and axis 2  -> (2, 4, 3)
print("x.transpose(1, 2)", xt)

print("same storage :", x.data_ptr() == xt.data_ptr())  # True -> it's a view
print("xt strides   :", xt.stride())

# permute generalises transpose to any reordering:
print("x.permute(2, 0, 1)", x.permute(2, 0, 1))  # (4, 2, 3)

x tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])

x.transpose(1, 2) tensor([[[ 0,  4,  8],
         [ 1,  5,  9],
         [ 2,  6, 10],
         [ 3,  7, 11]],

        [[12, 16, 20],
         [13, 17, 21],
         [14, 18, 22],
         [15, 19, 23]]])
same storage : True
xt strides   : (12, 1, 4)
x.permute(2, 0, 1) tensor([[[ 0,  4,  8],
         [12, 16, 20]],

        [[ 1,  5,  9],
         [13, 17, 21]],

        [[ 2,  6, 10],
         [14, 18, 22]],

        [[ 3,  7, 11],
         [15, 19, 23]]])


## 3. `squeeze` — remove size-1 axes

`squeeze()` with no argument drops **every** axis of size 1.
`squeeze(dim)` drops only that axis, and *only if* its size is 1 — otherwise it
is a harmless no-op. This is handy for cleaning up stray length-1 dimensions
left behind by reductions or indexing.

squeeze will not do anything if there is no redundant dimention

In [7]:
a = torch.zeros(2, 3, 2)
show("a", a)
show("a.squeeze()", a.squeeze())     # (3,)    -> all size-1 axes gone
show("a.squeeze(0)", a.squeeze(0))   # (3, 1)  -> only axis 0 dropped
show("a.squeeze(2)", a.squeeze(2))   # (1, 3)  -> only axis 2 dropped
show("a.squeeze(1)", a.squeeze(1))   # (1, 3, 1) no-op: axis 1 has size 3

a
tensor([[[0., 0.],
         [0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.],
         [0., 0.]]])
torch.Size([2, 3, 2])

a.squeeze()
tensor([[[0., 0.],
         [0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.],
         [0., 0.]]])
torch.Size([2, 3, 2])

a.squeeze(0)
tensor([[[0., 0.],
         [0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.],
         [0., 0.]]])
torch.Size([2, 3, 2])

a.squeeze(2)
tensor([[[0., 0.],
         [0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.],
         [0., 0.]]])
torch.Size([2, 3, 2])

a.squeeze(1)
tensor([[[0., 0.],
         [0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.],
         [0., 0.]]])
torch.Size([2, 3, 2])



## 4. `unsqueeze` — insert a size-1 axis

`unsqueeze(dim)` is the inverse of `squeeze`: it inserts a new axis of size 1 at
position `dim`. It is exactly equivalent to indexing with `None`. You reach for
this constantly when you need shapes to **broadcast** — e.g. adding an attention
mask of shape `(B, 1, 1, L)` to scores of shape `(B, h, L, L)`.

In [8]:
v = torch.tensor([[10, 20, 30],[11, 12, 13]])
show("v", v)                          # (3,)
show("v.unsqueeze(0)", v.unsqueeze(0))    # (1, 3) -> row vector
show("v.unsqueeze(1)", v.unsqueeze(1))    # (3, 1) -> column vector
show("v.unsqueeze(-1)", v.unsqueeze(-1))  # (3, 1) -> insert at the end
show("v.unsqueeze(2)", v.unsqueeze(2))
# Indexing with None does the same thing:
show("v[None, :]", v[None, :])        # (1, 3)
show("v[:, None]", v[:, None])        # (3, 1)

v
tensor([[10, 20, 30],
        [11, 12, 13]])
torch.Size([2, 3])

v.unsqueeze(0)
tensor([[[10, 20, 30],
         [11, 12, 13]]])
torch.Size([1, 2, 3])

v.unsqueeze(1)
tensor([[[10, 20, 30]],

        [[11, 12, 13]]])
torch.Size([2, 1, 3])

v.unsqueeze(-1)
tensor([[[10],
         [20],
         [30]],

        [[11],
         [12],
         [13]]])
torch.Size([2, 3, 1])

v.unsqueeze(2)
tensor([[[10],
         [20],
         [30]],

        [[11],
         [12],
         [13]]])
torch.Size([2, 3, 1])

v[None, :]
tensor([[[10, 20, 30],
         [11, 12, 13]]])
torch.Size([1, 2, 3])

v[:, None]
tensor([[[10, 20, 30]],

        [[11, 12, 13]]])
torch.Size([2, 1, 3])



## 5. The `dim` / `axis` argument in reductions

Operations like `sum`, `mean`, `max`, and `softmax` take a `dim`: that axis is
the one **collapsed** (for reductions) or **normalized over** (for softmax).
The result drops that axis. `dim=-1` means the last axis, which is the most
common choice because it is robust to extra leading batch dimensions.

YOU ADD AS MANY NUMBERS AS ARE IN A DIMENTION YOU ARE COLLAPSING

In [9]:
x = torch.arange(24, dtype=torch.float).reshape(2, 3, 4)
show("x", x)
show("x.sum(dim=0)", x.sum(dim=0))    # (3, 4) -> axis 0 collapsed
show("x.sum(dim=1)", x.sum(dim=1))    # (2, 4) -> axis 1 collapsed
show("x.sum(dim=-1)", x.sum(dim=-1))  # (2, 3) -> last axis collapsed
print("dim=-1 == dim=2 here:", torch.equal(x.sum(dim=-1), x.sum(dim=2)))

# softmax(dim=-1): each group along the LAST axis becomes a distribution.
w = torch.softmax(x, dim=-1)
print("\nsoftmax over dim=-1; each group of 4 sums to 1:")
print(w.sum(dim=-1))                  # all 1.0

x
tensor([[[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.]],

        [[12., 13., 14., 15.],
         [16., 17., 18., 19.],
         [20., 21., 22., 23.]]])
torch.Size([2, 3, 4])

x.sum(dim=0)
tensor([[12., 14., 16., 18.],
        [20., 22., 24., 26.],
        [28., 30., 32., 34.]])
torch.Size([3, 4])

x.sum(dim=1)
tensor([[12., 15., 18., 21.],
        [48., 51., 54., 57.]])
torch.Size([2, 4])

x.sum(dim=-1)
tensor([[ 6., 22., 38.],
        [54., 70., 86.]])
torch.Size([2, 3])

dim=-1 == dim=2 here: True

softmax over dim=-1; each group of 4 sums to 1:
tensor([[1., 1., 1.],
        [1., 1., 1.]])


In [24]:
print(x.size(0))
print(x.size(1))
print(x.size(2))

2
3
4


In [15]:
vecten = torch.tensor([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
viewed = vecten.view(2, 5)             # (2, 5)
print("\nviewed", viewed)


#vecten.view(2,4) will crash because 2*4 != 10


viewed tensor([[ 1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10]])


In [21]:
print(vecten.size(0))
print(vecten.reshape(2,5))
#print(vecten.size(1))

10
tensor([[ 1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10]])


## 6. Putting it together: the attention shape pipeline

This is exactly what the multi-head attention class does, in shapes.
Watch `view`, `transpose(1, 2)`, and `softmax(dim=-1)` all appear.

(batch, sequence length, embed dim for all) 

            is transformed to

(batch, heads, sequence length, embed per head)

In [ ]:
B, L, E, h = 2, 5, 32, 4      # batch, seq len, embed dim, heads
d = E // h                     # per-head dim = 8

tok = torch.randn(B, L, E)                 # output of q_proj
show("after q_proj", tok)                  # (2, 5, 32)

heads = tok.view(B, L, h, d)               # split last axis into (h, d)
show("view (B,L,h,d)", heads)              # (2, 5, 4, 8)

heads = heads.transpose(1, 2)              # head axis up front
show("transpose(1,2)", heads)              # (2, 4, 5, 8)
                                           # (batch, heads, sequence length, embed per head)    

scores = heads @ heads.transpose(2, 3) / d ** 0.5   # batched per-head matmul
show("scores (B,h,L,L)", scores)           # (2, 4, 5, 5)

attn = scores.softmax(dim=-1)              # normalize over the key axis
print("each query's weights sum to 1:",
      torch.allclose(attn.sum(dim=-1), torch.ones(B, h, L)))

after q_proj
tensor([[[-1.1799,  0.4436,  0.9242, -0.2112, -1.0144,  0.4836,  0.0198,
          -0.5928, -0.9103, -0.8782,  1.2003,  0.9978, -0.2616,  0.1428,
          -2.0798, -0.8175,  0.1692, -0.7459,  0.5359,  0.7780, -0.5135,
          -1.4313, -0.1422, -0.8498,  0.0123, -0.1328, -1.6957,  1.6012,
          -1.9430, -0.3443,  1.8942,  0.4024],
         [-0.1724,  1.1510,  1.0670,  0.9069,  0.1235,  1.1161,  1.6699,
           1.2976, -0.4427,  0.5064, -0.6463,  0.3324, -0.2821,  0.7071,
           0.9358,  0.8842, -1.7123,  1.9830, -0.5988, -0.1118,  0.1418,
           0.5198, -0.7694, -0.2799,  0.4811,  0.4955,  0.9556, -0.4961,
           1.7223, -0.8042, -1.2699, -0.3781],
         [-0.0128, -0.0659,  0.2581, -0.6941,  0.2459, -0.2589,  0.9904,
           0.8446,  1.4652,  0.1282, -0.8016, -2.2738, -0.9700, -0.7394,
          -1.1096, -0.8232, -1.3888, -0.1753, -0.4524,  1.5289,  1.1248,
          -0.0224, -0.4459,  0.2623,  1.4676,  1.8098, -0.7917, -0.5094,
          -0.4361